In [121]:
# Task 8: RDD-Based Implementation
# Using RDD API:
# 1. Calculate total confirmed per country.
# 2. Calculate total deaths per country.
# 3. Compute death percentage using reduceByKey.
# 4. Compare RDD performance vs DataFrame.
# Explain:
# 1. Why reduceByKey is preferred over groupByKey
# 2. When RDD should be avoided

In [122]:
import time
from pyspark.sql import *

In [123]:
spark = SparkSession.builder\
    .appName("RDD Based implementation")\
    .master("yarn")\
.getOrCreate()
spark

26/02/23 11:33:30 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


### Reading Data

In [124]:
rdd_full_grouped = spark.sparkContext.textFile("hdfs:///data/covid/raw/full_grouped.csv")

In [125]:
header = rdd_full_grouped.first()
rdd_full_grouped = rdd_full_grouped.filter(lambda line: line != header)

### 1. Calculate total confirmed per country.

In [126]:
rdd_country_confirmed = rdd_full_grouped.map(lambda line: line.split(",")) \
    .map(lambda columns: (columns[1], int(columns[2]))) \
    .reduceByKey(lambda confirmed1, confirmed2: confirmed1 + confirmed2)

In [127]:
result = rdd_country_confirmed.collect()
print("Country wise confirmed:")
print("----------------------------")
print("Country -> Confirmed cases")
print("----------------------------")
for country, confirmed in result:
    print(f"{country} -> {confirmed}")

[Stage 1:=============================>                             (1 + 1) / 2]

Country wise confirmed:
----------------------------
Country -> Confirmed cases
----------------------------
Afghanistan -> 1936390
Albania -> 196702
Andorra -> 94404
Angola -> 22662
Antigua and Barbuda -> 4487
Australia -> 960247
Bahamas -> 12100
Bahrain -> 1755206
Bangladesh -> 8754729
Barbados -> 10652
Benin -> 64406
Bhutan -> 4971
Bolivia -> 2170351
Botswana -> 15306
Brunei -> 18168
Bulgaria -> 410722
Burkina Faso -> 96153
Cabo Verde -> 82732
Cambodia -> 17079
Chile -> 16935654
China -> 14132002
Costa Rica -> 347151
Cote d'Ivoire -> 611062
Cuba -> 216346
Cyprus -> 107176
Djibouti -> 336216
Dominican Republic -> 2495433
Ecuador -> 4678496
Egypt -> 4142819
El Salvador -> 453036
Estonia -> 216505
Eswatini -> 63160
Ethiopia -> 357928
Fiji -> 2266
France -> 21210926
Gabon -> 330678
Germany -> 21059152
Guyana -> 19089
Haiti -> 333181
Holy See -> 1356
Honduras -> 1228583
Hungary -> 396247
Iceland -> 221241
Israel -> 2677930
Italy -> 26745145
Japan -> 1952495
Kenya -> 464603
Kosovo -> 2259

In [129]:
df_country_confirmed = rdd_country_confirmed.map(
    lambda row: Row(Country=row[0], Deaths=row[1])
).toDF()

df_country_confirmed.write \
    .mode("overwrite") \
    .option("header", True) \
    .parquet("/data/covid/staging/rdd_countrywise_confirmed")

### 1. Calculate total deaths per country.

In [130]:
rdd_country_deaths = rdd_full_grouped.map(lambda line: line.split(",")) \
    .map(lambda columns: (columns[1], int(columns[3]))) \
    .reduceByKey(lambda deaths1, deaths2: deaths1 + deaths2)

In [131]:
result = rdd_country_deaths.collect()
print("Country wise Deaths:")
print("----------------------------")
print("Country -> Deaths")
print("----------------------------")
for country, deaths in result:
    print(f"{country} -> {deaths}")

Country wise Deaths:
----------------------------
Country -> Deaths
----------------------------
Venezuela -> 4101
Western Sahara -> 63
Afghanistan -> 49098
Albania -> 5708
Andorra -> 5423
Angola -> 1078
Antigua and Barbuda -> 326
Australia -> 11387
Bahamas -> 1203
Bahrain -> 5115
Bangladesh -> 115633
Barbados -> 738
Benin -> 1095
Bhutan -> 0
Bolivia -> 78032
Botswana -> 120
Brunei -> 226
Bulgaria -> 17654
Burkina Faso -> 5583
Cabo Verde -> 854
Cambodia -> 0
Chile -> 322480
China -> 672413
Costa Rica -> 2037
Cote d'Ivoire -> 4717
Cuba -> 8145
Cyprus -> 1962
Djibouti -> 3011
Dominican Republic -> 61786
Ecuador -> 346618
Egypt -> 186936
El Salvador -> 11429
Estonia -> 6826
Eswatini -> 763
Ethiopia -> 5887
Fiji -> 0
France -> 3048524
Gabon -> 2497
Germany -> 871322
Guyana -> 1346
Haiti -> 6663
Holy See -> 0
Honduras -> 37941
Hungary -> 51053
Iceland -> 1141
Israel -> 31627
Italy -> 3707717
Japan -> 85559
Kenya -> 10463
Kosovo -> 5229
Kuwait -> 23084
Laos -> 0
Lebanon -> 3520
Libya -> 1612

In [132]:
df_country_deaths = rdd_country_deaths.map(
    lambda row: Row(Country=row[0], Deaths=row[1])
).toDF()

df_country_deaths.write \
    .mode("overwrite") \
    .option("header", True) \
    .parquet("/data/covid/staging/rdd_countrywise_deaths")

### 3. Compute death percentage using reduceByKey.

In [133]:
confirmed_and_deaths_joined = rdd_country_confirmed.join(rdd_country_deaths)

In [134]:
rdd_death_percentage_per_country = confirmed_and_deaths_joined.map(
    lambda country_data: (
        country_data[0],
        round(
            (country_data[1][1] / country_data[1][0]) * 100,
            2
        ) if country_data[1][0] > 0 else 0
    )
)

In [135]:
result = rdd_death_percentage_per_country.collect()
print("Country wise death percentage:")
print("----------------------------")
print("Country -> Death Percentage")
print("----------------------------")
for country, death_percentage in result:
    print(f"{country} -> {death_percentage}")

Country wise death percentage:
----------------------------
Country -> Death Percentage
----------------------------
Venezuela -> 0.99
Western Sahara -> 6.99
Afghanistan -> 2.54
Albania -> 2.9
Andorra -> 5.74
Angola -> 4.76
Antigua and Barbuda -> 7.27
Australia -> 1.19
Bahamas -> 9.94
Bahrain -> 0.29
Bangladesh -> 1.32
Barbados -> 6.93
Benin -> 1.7
Bhutan -> 0.0
Bolivia -> 3.6
Botswana -> 0.78
Brunei -> 1.24
Bulgaria -> 4.3
Burkina Faso -> 5.81
Cabo Verde -> 1.03
Cambodia -> 0.0
Chile -> 1.9
China -> 4.76
Costa Rica -> 0.59
Cote d'Ivoire -> 0.77
Cuba -> 3.76
Cyprus -> 1.83
Djibouti -> 0.9
Dominican Republic -> 2.48
Ecuador -> 7.41
Egypt -> 4.51
El Salvador -> 2.52
Estonia -> 3.15
Eswatini -> 1.21
Ethiopia -> 1.64
Fiji -> 0.0
France -> 14.37
Gabon -> 0.76
Germany -> 4.14
Guyana -> 7.05
Haiti -> 2.0
Holy See -> 0.0
Honduras -> 3.09
Hungary -> 12.88
Iceland -> 0.52
Israel -> 1.18
Italy -> 13.86
Japan -> 4.38
Kenya -> 2.25
Kosovo -> 2.31
Kuwait -> 0.74
Laos -> 0.0
Lebanon -> 2.11
Libya -> 

In [136]:
df_death_percentage_per_country = rdd_death_percentage_per_country.map(
    lambda row: Row(Country=row[0], Death_Percentage=row[1])
).toDF()

df_death_percentage_per_country.write \
    .mode("overwrite") \
    .option("header", True) \
    .parquet("/data/covid/staging/rdd_death_percentage_countrywise")

### 4. Compare RDD performance vs DataFrame.

In [137]:
start_time = time.time()
rdd_full_grouped = spark.sparkContext.textFile("hdfs:///data/covid/raw/full_grouped.csv")
header = rdd_full_grouped.first()
rdd_full_grouped = rdd_full_grouped.filter(lambda line: line != header)
rdd_country_confirmed = rdd_full_grouped.map(lambda line: line.split(",")) \
    .map(lambda columns: (columns[1], int(columns[2]))) \
    .reduceByKey(lambda confirmed1, confirmed2: confirmed1 + confirmed2)
rdd_country_confirmed.count()
rdd_time = time.time() - start_time
print(f"RDD Execution Time: {rdd_time:.2f} seconds")

start_time = time.time()
df_full_grouped = spark.read.csv("hdfs:///data/covid/raw/full_grouped.csv",header=True,inferSchema=True)
df_country_confirmed = df_full_grouped.groupBy("Country/Region").sum("Confirmed")
df_country_confirmed.count()
df_time = time.time() - start_time
print(f"DataFrame Execution Time: {df_time:.2f} seconds")

if rdd_time > df_time:
    print("DataFrame is faster")
else:
    print("RDD is faster")

RDD Execution Time: 0.25 seconds


DataFrame Execution Time: 1.35 seconds
RDD is faster


### 1. Why reduceByKey is preferred over groupByKey
```
groupByKey()
* Sends all values of a key across the network
* Then performs aggregation
* No pre-aggregation on mapper side

reduceByKey()
* Performs local aggregation first (map-side combine)
* Then shuffles only reduced results
* Much less network traffic
```

### 2. When RDD should be avoided
```
1 Structured Data Exists
2 You Need Performance Optimization
3 Large Aggregations / Joins
4 You Need SQL Capability
5 Production Systems
```

In [138]:
spark.stop()